In [1]:
import pandas as pd
import pprint
import tabulate

In [2]:
tripData = pd.read_parquet('../data/yellow_tripdata_2019-06.parquet')
tripData.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2019-06-01 00:55:13,2019-06-01 00:56:17,1.0,0.0,1.0,N,145,145,2,3.0,0.5,0.5,0.00,0.0,0.3,4.30,0.0,None
1,1,2019-06-01 00:06:31,2019-06-01 00:06:52,1.0,0.0,1.0,N,262,263,2,2.5,3.0,0.5,0.00,0.0,0.3,6.30,2.5,None
2,1,2019-06-01 00:17:05,2019-06-01 00:36:38,1.0,4.4,1.0,N,74,7,2,17.5,0.5,0.5,0.00,0.0,0.3,18.80,0.0,None
3,1,2019-06-01 00:59:02,2019-06-01 00:59:12,0.0,0.8,1.0,N,145,145,2,2.5,1.0,0.5,0.00,0.0,0.3,4.30,0.0,None
4,1,2019-06-01 00:03:25,2019-06-01 00:15:42,1.0,1.7,1.0,N,113,148,1,9.5,3.0,0.5,2.65,0.0,0.3,15.95,2.5,None


In [3]:
zoneData = pd.read_csv('../data/taxi_zone_lookup.csv')
zoneData.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [4]:
tripData['tpep_pickup_datetime']

0         2019-06-01 00:55:13
1         2019-06-01 00:06:31
2         2019-06-01 00:17:05
3         2019-06-01 00:59:02
4         2019-06-01 00:03:25
                  ...        
6971555   2019-06-30 23:35:00
6971556   2019-06-30 23:04:06
6971557   2019-06-30 23:34:24
6971558   2019-06-30 23:52:00
6971559   2019-06-30 23:29:00
Name: tpep_pickup_datetime, Length: 6971560, dtype: datetime64[us]

In [5]:
def extract_time(df, datetimeCol):

    df[f'{datetimeCol}_time'] = df[datetimeCol].dt.time
    return df

def bucketize_time(df, datetimeCol, timeInterval):

    seconds_from_midnight = (
    df[datetimeCol].dt.hour * 3600 + 
    df[datetimeCol].dt.minute * 60 + 
    df[datetimeCol].dt.second
)
    df[f'{datetimeCol}_bucket_id'] = seconds_from_midnight // (timeInterval * 60)

    df = df[df['fare_amount'] > 0]

    return df

# processedTripData = extract_time(tripData, 'tpep_pickup_datetime')
# processedTripData = extract_time(processedTripData, 'tpep_dropoff_datetime')

processedTripData = bucketize_time(tripData, 'tpep_pickup_datetime', 5)
processedTripData = bucketize_time(processedTripData, 'tpep_dropoff_datetime', 5)
processedTripData[['PULocationID', 'DOLocationID', 'tpep_pickup_datetime', 'tpep_pickup_datetime_bucket_id',
    'tpep_dropoff_datetime', 'tpep_dropoff_datetime_bucket_id']].head()

,PULocationID,DOLocationID,tpep_pickup_datetime,tpep_pickup_datetime_bucket_id,tpep_dropoff_datetime,tpep_dropoff_datetime_bucket_id
0,145,145,2019-06-01 00:55:13,11,2019-06-01 00:56:17,11
1,262,263,2019-06-01 00:06:31,1,2019-06-01 00:06:52,1
2,74,7,2019-06-01 00:17:05,3,2019-06-01 00:36:38,7
3,145,145,2019-06-01 00:59:02,11,2019-06-01 00:59:12,11
4,113,148,2019-06-01 00:03:25,0,2019-06-01 00:15:42,3


In [10]:
def get_zoneName_zoneID(df, columnName, zoneName):

    df = df[df[columnName] == zoneName]

    return df['LocationID']

manhattanZoneIds = get_zoneName_zoneID(zoneData, 'Borough', 'Manhattan')
manhattanZoneIds

3        4
11      12
12      13
23      24
40      41
      ... 
245    246
248    249
260    261
261    262
262    263
Name: LocationID, Length: 69, dtype: int64

In [12]:
def filter_zone_data(df, zoneSeries):

    filtered_df = df[df['PULocationID'].isin(zoneSeries) | df['DOLocationID'].isin(zoneSeries)]
    return filtered_df

processedTripData = filter_zone_data(processedTripData, manhattanZoneIds)
processedTripData

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,tpep_pickup_datetime_bucket_id,tpep_dropoff_datetime_bucket_id,duration_seconds
1,1,2019-06-01 00:06:31,2019-06-01 00:06:52,1.0,0.00,1.0,N,262,263,2,...,0.5,0.00,0.00,0.3,6.30,2.5,None,1,1,21.0
2,1,2019-06-01 00:17:05,2019-06-01 00:36:38,1.0,4.40,1.0,N,74,7,2,...,0.5,0.00,0.00,0.3,18.80,0.0,None,3,7,1173.0
4,1,2019-06-01 00:03:25,2019-06-01 00:15:42,1.0,1.70,1.0,N,113,148,1,...,0.5,2.65,0.00,0.3,15.95,2.5,None,0,3,737.0
5,1,2019-06-01 00:28:31,2019-06-01 00:39:23,2.0,1.60,1.0,N,79,125,1,...,0.5,1.00,0.00,0.3,14.30,2.5,None,5,7,652.0
6,1,2019-06-01 00:46:46,2019-06-01 00:50:55,4.0,0.60,1.0,N,211,148,2,...,0.5,0.00,0.00,0.3,8.30,2.5,None,9,10,249.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6971550,2,2019-06-30 23:54:52,2019-07-01 00:16:20,NaN,5.08,NaN,NaN,151,68,0,...,0.0,0.00,0.00,0.3,32.50,NaN,None,286,3,1288.0
6971551,2,2019-06-30 23:14:33,2019-06-30 23:52:02,NaN,9.72,NaN,NaN,142,95,0,...,0.0,0.00,0.00,0.3,49.20,NaN,None,278,286,2249.0
6971553,2,2019-06-30 23:22:18,2019-07-01 00:03:56,NaN,21.00,NaN,NaN,237,77,0,...,0.0,0.00,6.12,0.3,44.14,NaN,None,280,0,2498.0
6971554,2,2019-06-30 23:46:19,2019-07-01 00:09:43,NaN,12.65,NaN,NaN,142,185,0,...,0.0,0.00,0.00,0.3,43.26,NaN,None,285,1,1404.0


In [13]:
def get_zone_id(df, zoneName):
    print(df[df['Borough'] == zoneName]['LocationID'])
    return df[df['Borough'] == zoneName]['LocationID']

def get_time_bucket_id(timeBucket):
    return timeBucket

def get_demand(df, zoneId, timeBucket):

    bucektId = get_time_bucket_id(timeBucket)

    return len(df[(df['PULocationID'] == zoneId) & (df['tpep_pickup_datetime_bucket_id'] == bucektId)])

demand = get_demand(processedTripData, 145, 130)
demand

11

In [13]:
def get_destinations(df, zoneId, timeBucket):

    bucektId = get_time_bucket_id(timeBucket)
    dfDest = df[(df['PULocationID'] == zoneId) & (df['tpep_pickup_datetime_bucket_id'] == bucektId)]

    return dfDest['DOLocationID'].value_counts(normalize = True)

destinationProbabilityDist = get_destinations(processedTripData, 145, 130)
destinationProbabilityDist

DOLocationID
145    0.500000
164    0.078947
193    0.052632
234    0.026316
140    0.026316
230    0.026316
186    0.026316
132    0.026316
48     0.026316
80     0.026316
226    0.026316
100    0.026316
255    0.026316
264    0.026316
233    0.026316
95     0.026316
162    0.026316
Name: proportion, dtype: float64

In [14]:
def get_fares(df, zoneId, timeBucket, destinationId):

    bucektId = get_time_bucket_id(timeBucket)
    dfFare = df[(df['PULocationID'] == zoneId) & (df['tpep_pickup_datetime_bucket_id'] == bucektId) & (df['DOLocationID'] == destinationId)]
    print(dfFare.to_markdown())
    
    # dfFare = dfFare[dfFare['fare_amount'] > 0]
    return dfFare['fare_amount'].median()

get_fares(processedTripData, 145, 130, 145)

|         |   VendorID | tpep_pickup_datetime   | tpep_dropoff_datetime   |   passenger_count |   trip_distance |   RatecodeID | store_and_fwd_flag   |   PULocationID |   DOLocationID |   payment_type |   fare_amount |   extra |   mta_tax |   tip_amount |   tolls_amount |   improvement_surcharge |   total_amount |   congestion_surcharge | airport_fee   |   tpep_pickup_datetime_bucket_id |   tpep_dropoff_datetime_bucket_id |
|--------:|-----------:|:-----------------------|:------------------------|------------------:|----------------:|-------------:|:---------------------|---------------:|---------------:|---------------:|--------------:|--------:|----------:|-------------:|---------------:|------------------------:|---------------:|-----------------------:|:--------------|---------------------------------:|----------------------------------:|
|   60588 |          1 | 2019-06-01 10:52:36    | 2019-06-01 10:52:42     |                 1 |            0    |            1 | N              

np.float64(2.5)

In [ ]:
def get_time_duration(df, zoneId, timeBucket, destinationId):

    bucektId = get_time_bucket_id(timeBucket)
    dfDuration = df[(df['PULocationID'] == zoneId) & (df['tpep_pickup_datetime_bucket_id'] == bucektId) & (df['DOLocationID'] == destinationId)].copy()
    dfDuration['duration_seconds'] = (dfDuration['tpep_dropoff_datetime'] - dfDuration['tpep_pickup_datetime']).dt.total_seconds()

    print(dfDuration.to_markdown())

    return dfDuration['duration_seconds'].median()

get_time_duration(processedTripData, 145, 130, 145)

|         |   VendorID | tpep_pickup_datetime   | tpep_dropoff_datetime   |   passenger_count |   trip_distance |   RatecodeID | store_and_fwd_flag   |   PULocationID |   DOLocationID |   payment_type |   fare_amount |   extra |   mta_tax |   tip_amount |   tolls_amount |   improvement_surcharge |   total_amount |   congestion_surcharge | airport_fee   |   tpep_pickup_datetime_bucket_id |   tpep_dropoff_datetime_bucket_id |   duration_seconds |
|--------:|-----------:|:-----------------------|:------------------------|------------------:|----------------:|-------------:|:---------------------|---------------:|---------------:|---------------:|--------------:|--------:|----------:|-------------:|---------------:|------------------------:|---------------:|-----------------------:|:--------------|---------------------------------:|----------------------------------:|-------------------:|
|   60588 |          1 | 2019-06-01 10:52:36    | 2019-06-01 10:52:42     |                 1 |       

np.float64(14.0)

In [14]:
def prepare_data(tripData):

    # get trip duration in seconds

    df = processedTripData
    df['duration_seconds'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds()

    # get demand count

    demand = (df.groupby(['PULocationID', 'tpep_pickup_datetime_bucket_id'])
            .size()
            .rename('demand'))
    
    # get destination distribution

    destCounts = (df.groupby(['PULocationID', 'tpep_pickup_datetime_bucket_id', 'DOLocationID'])
                .size())
    destDist = (destCounts / destCounts.groupby(level=[0, 1]).transform('sum')).rename('prob')

    # get fare and duration stats (median)

    odStats = (df[df['fare_amount'] > 0]
             .groupby(['PULocationID', 'DOLocationID'])
             .agg(fare_median     = ('fare_amount',      'median'),
                  duration_median = ('duration_seconds', 'median'),
                  n_trips         = ('fare_amount',       'size')))

    return demand, destDist, odStats


demand, destDist, odStats = prepare_data(processedTripData)

In [15]:
demand

PULocationID  tpep_pickup_datetime_bucket_id
1             97                                1
              125                               1
              158                               1
              177                               1
              185                               1
                                               ..
265           283                               2
              284                               1
              285                               4
              286                               2
              287                               4
Name: demand, Length: 36305, dtype: int64

In [16]:
destDist

PULocationID  tpep_pickup_datetime_bucket_id  DOLocationID
1             97                              186             1.00
              125                             186             1.00
              158                             162             1.00
              177                             238             1.00
              185                             231             1.00
                                                              ... 
265           286                             142             0.50
              287                             144             0.25
                                              148             0.25
                                              234             0.25
                                              263             0.25
Name: prob, Length: 993873, dtype: float64

In [17]:
odStats

fare_median  duration_median  n_trips
PULocationID DOLocationID                                       
1            158                 30.00           1837.0        2
             161                  8.50            579.0        1
             162                 55.00           2804.0        1
             163                 75.00           3866.0        1
             186                 34.75           1049.0        2
...                                ...              ...      ...
265          246                 13.50           1074.0        8
             249                  6.00            332.0       17
             261                 13.75           1030.5        4
             262                  9.50            577.0        3
             263                 16.00            960.0        4

[20375 rows x 3 columns]

In [1]:
import numpy as np

In [6]:
demand = np.load('../data/artifacts/demand.npy', allow_pickle=True)
demand[0,:]

array([53., 71., 61., 69., 56., 68., 77., 71., 59., 75., 62., 73., 75.,
       63., 64., 64., 59., 93., 55., 62., 60., 64., 60., 49., 62., 58.,
       59., 48., 69., 36., 51., 40., 44., 38., 31., 36., 32., 34., 36.,
       27., 20., 21., 26., 14., 22., 18., 27., 25., 24., 23., 13., 17.,
       15.,  8., 10., 12.,  9., 12., 14.,  5.,  8., 10.,  7.,  9.,  6.,
       10.,  3.,  6.,  6., 19.,  7., 12.,  8., 14.,  9., 10., 14., 14.,
       16., 12., 20., 20., 24., 16., 14., 12., 21., 24., 28., 23., 32.,
       32., 39., 33., 34., 41., 38., 37., 31., 48., 49., 51., 50., 59.,
       36., 49., 49., 50., 38., 45., 34., 54., 46., 43., 39., 44., 40.,
       37., 45., 37., 41., 43., 44., 42., 36., 22., 32., 37., 45., 35.,
       31., 31., 26., 31., 37., 29., 37., 23., 32., 33., 27., 26., 32.,
       22., 27., 24., 31., 32., 35., 13., 28., 27., 35., 36., 31., 32.,
       37., 32., 35., 36., 30., 30., 33., 34., 23., 27., 35., 27., 33.,
       32., 33., 25., 29., 28., 28., 26., 33., 33., 37., 36., 29

In [9]:
import json

with open('../data/artifacts/id2idx.json', 'r') as f:
    id2idx = json.load(f)

with open('../data/artifacts/idx2id.json', 'r') as f:
    idx2id = json.load(f)

with open('../data/artifacts/adjacent_zones.json', 'r') as f:
    data = json.load(f)

data

{'4': [232, 148, 79, 224],
 '12': [88, 261, 13],
 '13': [12, 261, 231],
 '24': [43, 151, 41, 166],
 '41': [43, 75, 24, 74, 166, 152, 42],
 '42': [41, 74, 166, 152, 116, 120],
 '43': [163, 237, 142, 236, 75, 239, 238, 151, 24, 41],
 '45': [209, 232, 148, 231, 144],
 '48': [68, 246, 50, 100, 230, 163, 142, 143],
 '50': [246, 48, 142, 143],
 '68': [249, 158, 90, 186, 246, 100, 48],
 '74': [75, 194, 41, 42],
 '75': [262, 263, 236, 43, 41, 74],
 '79': [232, 148, 4, 224, 107, 234, 144, 114, 113],
 '87': [88, 209, 261],
 '88': [12, 87, 261],
 '90': [234, 113, 249, 68, 186],
 '100': [164, 68, 186, 161, 230, 48],
 '103': [],
 '104': [],
 '105': [],
 '107': [79, 224, 234, 137, 170, 164, 113],
 '113': [79, 107, 234, 114, 249, 90],
 '114': [148, 79, 144, 211, 125, 113, 249],
 '116': [152, 42, 244, 120],
 '120': [42, 116, 244, 243, 127],
 '125': [231, 211, 114, 249, 158],
 '127': [120, 243, 128],
 '128': [243, 127],
 '137': [224, 107, 170, 233],
 '140': [229, 141, 262, 263],
 '141': [162, 237, 229,

In [12]:
convertedZones = {}

for k in data:
    convertedZones[id2idx[str(k)]] = [id2idx[str(n)] for n in data[k]]
convertedZones

{0: [55, 35, 13, 51],
 1: [15, 66, 2],
 2: [1, 66, 54],
 3: [6, 36, 4, 44],
 4: [6, 12, 3, 11, 44, 37, 5],
 5: [4, 11, 44, 37, 24, 25],
 6: [42, 59, 32, 58, 12, 61, 60, 36, 3, 4],
 7: [49, 55, 35, 54, 34],
 8: [10, 64, 9, 17, 53, 42, 32, 33],
 9: [64, 8, 32, 33],
 10: [65, 39, 16, 46, 64, 17, 8],
 11: [12, 47, 4, 5],
 12: [67, 68, 58, 6, 4, 11],
 13: [55, 35, 0, 51, 21, 57, 34, 23, 22],
 14: [15, 49, 66],
 15: [1, 14, 66],
 16: [57, 22, 65, 10, 46],
 17: [43, 10, 46, 40, 53, 8],
 18: [],
 19: [],
 20: [],
 21: [13, 51, 57, 29, 45, 43, 22],
 22: [13, 21, 57, 23, 65, 16],
 23: [35, 13, 34, 50, 26, 22, 65],
 24: [37, 5, 63, 25],
 25: [5, 24, 63, 62, 27],
 26: [54, 50, 23, 65, 39],
 27: [25, 62, 28],
 28: [62, 27],
 29: [51, 21, 45, 56],
 30: [52, 31, 67, 68],
 31: [41, 59, 52, 30, 67, 68, 58],
 32: [9, 8, 42, 33, 6, 61],
 33: [9, 8, 32, 61],
 34: [7, 35, 13, 54, 50, 23],
 35: [7, 55, 0, 13, 34, 23],
 36: [6, 60, 3],
 37: [4, 44, 5, 24],
 38: [],
 39: [26, 65, 10, 64],
 40: [45, 43, 17, 41

In [13]:
maxLen = 0

for k in convertedZones:
    if len(convertedZones[k]) > maxLen:
        maxLen = len(convertedZones[k])
maxLen

10

In [14]:
convertedZones[58]

[59, 31, 68, 6, 12]